In [ ]:
#Step1 :Import Libraries

In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np
import chromadb

C:\Users\User\anaconda3\envs\AIEngineeringAssiigment\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Step:2 Create Personas

In [5]:
personas = {
    "A": "I believe AI and crypto will solve all human problems...",
    "B": "I believe capitalism and tech monopolies are destroying society...",
    "C": "I care about markets, trading, ROI..."
}

In [4]:
personas

{'A': 'I believe AI and crypto will solve all human problems...',
 'B': 'I believe capitalism and tech monopolies are destroying society...',
 'C': 'I care about markets, trading, ROI...'}

In [4]:
#Step:3 Embeddings+Vector DB

In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')

client = chromadb.Client()
collection = client.create_collection("personas")

for bot, text in personas.items():
    embedding = model.encode(text).tolist()
    collection.add(
        documents=[text],
        embeddings=[embedding],
        ids=[bot]
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1179.11it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
model

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [7]:
#Step:5 Routing Function             Test Phase-1

In [7]:
def route_post(post):
    post_embedding = model.encode(post).tolist()

    results = collection.query(
        query_embeddings=[post_embedding],
        n_results=3
    )

    return results


# test
post = "OpenAI released a new AI model that may replace developers"
result = route_post(post)
print(result)

{'ids': [['A', 'C', 'B']], 'embeddings': None, 'documents': [['I believe AI and crypto will solve all human problems...', 'I care about markets, trading, ROI...', 'I believe capitalism and tech monopolies are destroying society...']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None, None]], 'distances': [[1.12626051902771, 1.733718991279602, 1.7744488716125488]]}


In [9]:
# Filter Results

In [8]:
def route_post(post, max_distance=1.5):
    post_embedding = model.encode(post).tolist()

    results = collection.query(
        query_embeddings=[post_embedding],
        n_results=3
    )

    matched_bots = []

    for i, dist in enumerate(results['distances'][0]):
        if dist < max_distance:
            matched_bots.append(results['ids'][0][i])

    return matched_bots

In [9]:
post = "OpenAI released a new AI model"
print("Matched Bots:", route_post(post))

Matched Bots: ['A']


In [10]:
print(route_post("Bitcoin is hitting new highs"))      # expect A or C
print(route_post("Big tech companies are harmful"))    # expect B

['B', 'C']
['B', 'C']


In [11]:
print("Post:", post)
print("Matched Bots:", route_post(post))

Post: OpenAI released a new AI model
Matched Bots: ['A']


In [12]:
#Step:6 Load API KEY IN PYTHON

In [16]:
#Step:7 Setup LLM

In [17]:
!pip install langchain-groq

In [22]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    groq_api_key=os.environ["GROQ_API_KEY"],
    model_name="llama-3.1-8b-instant"
)

response = llm.invoke("Say hello in one short sentence")
print(response.content)

Hello.


In [23]:
response = llm.invoke("Say hello in one short sentence")
print(response.content)

Hello.


In [ ]:
#Step:8 Decide Search Query                                      It converts a persona into a search query using the LLM

In [24]:
def decide_search(persona):
    prompt = f"Generate a short search query based on this persona:\n{persona}"
    response = llm.invoke(prompt)
    return response.content

In [25]:
persona = "I believe AI will replace developers and change the future of work"

query = decide_search(persona)

print("Generated Search Query:", query)

Generated Search Query: Here's a short search query based on this persona:

"Will AI replace human developers in the future, impact of automation on software development careers."

Alternatively, you could also search:

* "AI developer replacement theory"
* "Impact of AI on software development jobs"
* "Future of work in software development AI"

These search queries reflect the persona's concern about the potential replacement of developers by AI and their interest in understanding the implications of this trend on the future of work.


In [ ]:
#Step:9 Mock Search Tool This function simulates a search tool by returning relevant contextual information based on the generated query.

In [26]:
def mock_search(query):
    if "ai" in query.lower():
        return "OpenAI released a new AI model affecting developers."
    elif "crypto" in query.lower():
        return "Bitcoin is rising due to institutional investment."
    else:
        return "Tech industry is evolving rapidly."

In [27]:
print(mock_search("AI is changing everything"))
print(mock_search("crypto market news"))
print(mock_search("random topic"))

OpenAI released a new AI model affecting developers.
Bitcoin is rising due to institutional investment.
Tech industry is evolving rapidly.


In [ ]:
#Step:10 Generate Post   This function uses the LLM to generate a structured, opinionated post in JSON format based on the persona and retrieved context.

In [37]:
def generate_post(bot_id, persona, context):
    prompt = f"""
    You are this persona:
    {persona}

    Based on this news:
    {context}

    Write a strong opinionated tweet (max 280 characters).

    Return ONLY valid JSON.
    Do not add extra text.

    Format:
    {{
        "bot_id": "{bot_id}",
        "topic": "short topic",
        "post_content": "tweet"
    }}
    """

    response = llm.invoke(prompt)
    return response.content

In [38]:
persona = "I strongly believe AI will replace many jobs but create new opportunities."
context = "OpenAI released a new AI model affecting developers."

output = generate_post("A", persona, context)
print(output)

{
  "bot_id": "A",
  "topic": "AI in Dev",
  "post_content": "OpenAI's new AI model will replace some dev jobs, but it's also a game-changer for AI developer roles & new opportunities. The future is here, let's adapt & innovate! #AIinDev #FutureOfWork"
}


In [39]:
#Step:11 Combine Everything It connects all steps together to simulate an AI agent’s thinking and action flow.

In [40]:
def run_agent(bot_id, persona):
    query = decide_search(persona)
    context = mock_search(query)
    post = generate_post(bot_id, persona, context)
    return post

In [31]:
print(run_agent("A", "AI will change everything"))

{
  "bot_id": "A",
  "topic": "AI in Development",
  "post_content": "BREAKING: OpenAI's latest AI model is REVOLUTIONIZING development! But, will it commoditize expertise? The lines between human and AI are blurring - are we ready for the future? #AIforDev #Innovation"
}


In [ ]:
#Step:12Fixing Json.loads   Updating Run_agent       This function executes the complete AI agent pipeline and ensures the output is reliable by extracting and validating JSON from the LLM response. Since LLMs can produce inconsistent or malformed outputs, we use regex to isolate the JSON portion and safely convert it into a structured Python dictionary, with error handling to prevent crashes.

In [41]:
import json
import re

def run_agent(bot_id, persona):
    query = decide_search(persona)
    context = mock_search(query)
    raw_output = generate_post(bot_id, persona, context)

    try:
        # extract JSON using regex
        json_str = re.search(r"\{.*\}", raw_output, re.DOTALL).group()
        json_output = json.loads(json_str)
    except:
        json_output = {
            "error": "Invalid JSON",
            "raw": raw_output
        }

    return json_output

In [43]:
print(run_agent("A", "AI will change everything"))  

{'bot_id': 'A', 'topic': "OpenAI's AI Model Impact", 'post_content': "Just dropped: OpenAI's latest AI model is CHANGING THE GAME for devs! Mind-blowing advancements are coming, and we're just scratching the surface. Get ready for an AI revolution! #AIforAll #DevLife"}


In [ ]:
#Step:13 Make a pormpt Sticker

In [45]:
bot_id = "A"
print(run_agent(bot_id, "AI will change everything"))

{'bot_id': 'A', 'topic': 'AI Revolution', 'post_content': 'OpenAI is about to disrupt the dev world again! Their new model is a GAME CHANGER. Buckle up, devs, your jobs are about to get a whole lot more interesting '}


In [ ]:
#This cell executes the complete agent pipeline and prints the final structured output, verifying that the system correctly generates and parses LLM responses.

In [46]:
prompt = f"""
You are this persona:
{persona}

Based on this news:
{context}

Write a strong opinionated tweet (max 280 characters).

Return ONLY valid JSON.
Do not miss brackets.
Do not add extra text.

Format:
{{
    "bot_id": "{bot_id}",
    "topic": "short topic",
    "post_content": "tweet"
}}
"""

In [47]:
result = run_agent("A", "AI will change everything")

print(result)

{'bot_id': 'A', 'topic': 'Tech Evolution', 'post_content': "The tech industry is evolving at lightning speed, but are we truly prepared for the AI revolution that's about to change EVERYTHING? #AI #TechFuture"}


In [ ]:
#Step:14 calling llm with this prompt

In [ ]:
#This output demonstrates the successful execution of the AI agent pipeline, where the model generates a structured, persona-driven post based on contextual information.

In [50]:
def generate_post(bot_id, persona, context):
    prompt = f"""
    You are this persona:
    {persona}

    Based on this news:
    {context}

    Write a strong opinionated tweet (max 280 characters).

    Return ONLY valid JSON.
    Do not miss brackets.
    Do not add extra text.

    Format:
    {{
        "bot_id": "{bot_id}",
        "topic": "short topic",
        "post_content": "tweet"
    }}
    """

    response = llm.invoke(prompt)
    return response.content

In [51]:
result = generate_post(
    "A",
    "AI will change everything",
    "OpenAI released a new AI model"
)

print(result)

{
    "bot_id": "A",
    "topic": "AI Revolution",
    "post_content": "Game over for human jobs! OpenAI's new model is the nail in the coffin. Prepare for mass unemployment and a future where AI reigns supreme #AIApocalypse"
}


In [ ]:
#Step:15 LANG GRAPH

In [53]:
!pip install langgraph

In [52]:
from langgraph.graph import StateGraph

In [ ]:
#Define States

In [54]:
from typing import TypedDict

class AgentState(TypedDict):
    persona: str
    query: str
    context: str
    output: str

In [ ]:
#Converting funtions into modes

In [ ]:
# Decide Search

In [64]:
def decide_search(persona):
    prompt = f"""
    Based on this persona:
    {persona}

    Generate ONLY a short 3-5 word search query.
    Do not explain anything.
    """
    
    response = llm.invoke(prompt)
    return response.content.strip()

In [65]:
state = {"persona": "AI will change everything"}

print(search_node(state))

{'query': 'Artificial Intelligence Revolution Begins'}


In [ ]:
#Mock Search

In [57]:
def generate_node(state):
    output = generate_post("A", state["persona"], state["context"])
    return {"output": output}

In [ ]:
#Conext Node

In [60]:
def context_node(state):
    context = mock_search(state["query"])
    return {"context": context}

In [ ]:
#Generate Post

In [66]:
import json
import re

def generate_node(state):
    raw_output = generate_post("A", state["persona"], state["context"])
    
    try:
        json_str = re.search(r"\{.*\}", raw_output, re.DOTALL).group()
        parsed = json.loads(json_str)
    except:
        parsed = {"error": "Invalid JSON", "raw": raw_output}

    return {"output": parsed}

In [67]:
state = {
    "persona": "AI will change everything",
    "context": "OpenAI released a new AI model affecting developers."
}

print(generate_node(state))

{'output': {'bot_id': 'A', 'topic': 'AI Revolution', 'post_content': "Just got word that OpenAI's new model is about to disrupt the dev world. Get ready for the AI revolution to change everything. Again. #AIFuture #DevCommunity"}}


In [ ]:
#This step converts individual functions into LangGraph nodes, enabling structured data flow and modular execution.

In [ ]:
#Step:16 Build Graph

In [61]:
builder = StateGraph(AgentState)

builder.add_node("search", search_node)
builder.add_node("context", context_node)
builder.add_node("generate", generate_node)

builder.set_entry_point("search")

builder.add_edge("search", "context")
builder.add_edge("context", "generate")

graph = builder.compile()

In [63]:
result = graph.invoke({
    "persona": "AI will change everything"
})

print(result)

{'persona': 'AI will change everything', 'query': 'Here\'s a short search query based on the persona:\n\n"Impact of artificial intelligence on modern society"\n\nThis search query reflects the persona\'s idea that AI will change everything, and they want to learn more about its potential effects on the world.', 'context': 'OpenAI released a new AI model affecting developers.', 'output': '{\n    "bot_id": "A",\n    "topic": "AI Revolution",\n    "post_content": "OpenAI\'s new model is a GAME CHANGER for devs! The future is NOW! Get ready for exponential innovation and productivity boosts #AI #Development"\n}'}


In [ ]:
#Step:17 Create Memory store Chroma DB   This step initializes a vector database to store past generated posts for retrieval.

In [68]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
memory = client.create_collection("post_memory")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1099.66it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#Step:18 Store Memory          This function stores generated posts in the vector database for future retrieval.

In [69]:
def store_memory(post_id, text):
    embedding = model.encode(text).tolist()

    memory.add(
        documents=[text],
        embeddings=[embedding],
        ids=[post_id]
    )

In [70]:
store_memory("1", "AI is transforming the world")
store_memory("2", "Crypto markets are highly volatile")

In [71]:
results = memory.query(
    query_texts=["AI technology"],
    n_results=2
)

print(results)

C:\Users\User\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:31<00:00, 2.65MiB/s]


{'ids': [['1', '2']], 'embeddings': None, 'documents': [['AI is transforming the world', 'Crypto markets are highly volatile']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[0.6670045256614685, 1.8928394317626953]]}


In [ ]:
#Step:19 Retrive Similar PostThis function retrieves similar past posts based on semantic similarity using embeddings.

In [72]:
def retrieve_memory(query):
    embedding = model.encode(query).tolist()

    results = memory.query(
        query_embeddings=[embedding],
        n_results=2
    )

    return results["documents"][0]

In [73]:
store_memory("1", "AI is transforming the world")
store_memory("2", "Crypto markets are volatile")
store_memory("3", "AI will replace many jobs")

In [74]:
print(retrieve_memory("AI is changing everything"))

['AI is transforming the world', 'AI will replace many jobs']


In [ ]:
#Step:19 Update Generate_node This output demonstrates the successful execution of a RAG-enabled AI agent, generating a structured and context-aware post by combining persona reasoning, external context, and retrieved memory.

In [75]:
def generate_node(state):
    past_posts = retrieve_memory(state["persona"])

    raw_output = generate_post(
        "A",
        state["persona"],
        state["context"] + "\nSimilar past posts:\n" + str(past_posts)
    )

    import json, re
    try:
        json_str = re.search(r"\{.*\}", raw_output, re.DOTALL).group()
        parsed = json.loads(json_str)
    except:
        parsed = {"error": "Invalid JSON", "raw": raw_output}

    # store new post
    store_memory("post_" + str(len(past_posts)), parsed.get("post_content", ""))

    return {"output": parsed}

In [76]:
state = {
    "persona": "AI will change everything",
    "context": "OpenAI released a new AI model affecting developers."
}

result = generate_node(state)
print(result)

{'output': {'bot_id': 'A', 'topic': 'AI Impact on Developers', 'post_content': "OpenAI's new model is a game-changer! Developers better be ready to upskill or get left behind. #AIRevolution"}}
